In [1]:
import torch
import pandas as pd
import re
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

In [2]:
torch.cuda.empty_cache()

In [3]:
base_model = "Qwen/Qwen2.5-7B-Instruct"
lora_path = "qwen_mcq_sft_model"

tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map={"": 0},   # 🔥 FIX
    trust_remote_code=True
)


model = PeftModel.from_pretrained(model, lora_path)

model.eval()


/home/sarmistha/miniconda3/envs/CIKM/lib/python3.9/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2SdpaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=3584, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
              )
              (k_proj): lora.Linear4bit(
                (base_layer): Linear4bi

In [4]:
def build_prompt(row):
    return f"""You are an expert MCQ solver.

Question: {row['question']}

Options:
a) {row['option_a']}
b) {row['option_b']}
c) {row['option_c']}
d) {row['option_d']}

INSTRUCTIONS:
- Choose exactly one: a, b, c, d
- If uncertain → idk
- Always provide reasoning

OUTPUT FORMAT (STRICT):
<answer>a/b/c/d/idk</answer>
<reasoning>2-3 line explanation</reasoning>
"""

In [5]:
def parse_output(output):
    final_answer = "idk"
    reasoning = ""

    ans_match = re.search(r"<answer>\s*(.*?)\s*</answer>", output, re.IGNORECASE)
    if ans_match:
        final_answer = ans_match.group(1).strip().lower()

    reason_match = re.search(r"<reasoning>\s*(.*?)\s*</reasoning>", output, re.IGNORECASE | re.DOTALL)
    if reason_match:
        reasoning = reason_match.group(1).strip()

    if final_answer not in ["a", "b", "c", "d", "idk"]:
        final_answer = "idk"

    if reasoning == "":
        final_answer = "idk"

    return final_answer, reasoning


In [6]:
df = pd.read_csv("final_combined_dataset.csv")


In [ ]:
from tqdm.auto import tqdm

results = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Running Inference"):
    
    prompt = build_prompt(row)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False
        )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_text = generated[len(prompt):]

    ans, reason = parse_output(generated_text)

    results.append({
        "question": row["question"],
        "pred_answer": ans,
        "pred_reasoning": reason
    })

Running Inference:   0%|          | 0/3521 [00:00<?, ?it/s]

/home/sarmistha/miniconda3/envs/CIKM/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/sarmistha/miniconda3/envs/CIKM/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:497: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/sarmistha/miniconda3/envs/CIKM/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:509: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


In [ ]:
out_df = pd.DataFrame(results)
out_df.to_csv("abstention_sft_qwen.csv", index=False)

print("Inference complete!")

In [12]:
out_df = pd.DataFrame(results)

df["pred_answer"] = out_df["pred_answer"]
df["pred_reasoning"] = out_df["pred_reasoning"]

df = df[
    [
        "item_id", "question", "final_answer",
        "option_a", "option_b", "option_c", "option_d",
        "question_level", "language", "reasoning",
        "pred_answer", "pred_reasoning"
    ]
]

df.to_csv("abstention_sft_qwen_final.csv", index=False)

print("Inference complete!")

Inference complete!
